In [35]:
import pandas as pd
import matplotlib as plt
import numpy as np
import datetime

df = pd.read_csv(
    r'C:\Users\tomas\New folder\Data Preprocessing\globalmart_dirty_orders_2000.csv', 
    keep_default_na=False,
    na_values=['-', '#N/A', 'N/A', 'NULL', '?', 'unknown', 'NONE', 'na', 'None', 'none', 'NaN', 'NA', 'nan', ''], # catch all null values before parsing
)   

df.index = df.index + 2 ## shifted index to align with csv

columns = [
    'record_id', 'source_system', 'customer_id', 'customer_name', 
    'customer_email', 'phone_raw', 'age_raw', 'gender_raw', 
    'signup_date_raw', 'order_id', 'order_date_raw', 'ship_date_raw', 
    'delivery_date_raw', 'customer_timezone', 'country_raw', 'city_raw', 
    'postal_code_raw', 'product_category_raw', 'product_name_raw', 
    'sku_raw', 'quantity_raw', 'unit_price_raw', 'currency_raw', 
    'discount_raw', 'shipping_cost_raw', 'item_weight_raw', 
    'payment_method_raw', 'order_status_raw', 'returned_flag_raw', 
    'return_reason_raw', 'satisfaction_score_raw', 'loyalty_points_raw', 
    'site_visits_last30_raw', 'support_tickets_90d_raw', 
    'distance_to_warehouse_km_raw', 'campaign_source_raw', 
    'customer_segment_raw', 'support_ticket_date_raw', 
    'complaint_text_raw', 'ingestion_batch', 'data_source_note'
]

*** TASK 1 ***

In [ ]:
def my_function(foo, bar):
    return foo+bar

def returnRows(df):
    return f"Total row count: {len(df)}"

def returnDatatype(df):
    return df.dtypes

def checkMissingValues(df, columns):
    null_rows = df[columns].isnull().sum() ## returns how many null rows are there in the series
    missing_values = null_rows[null_rows > 0] ## filters all boolean values that are greater than 0 to be passed through {missing_values} variable
    total_rows = len(df)
    
    percentages = (missing_values / total_rows) * 100
    print("--------------------Missing Rows--------------------")
    summary_df = pd.DataFrame({
        'Missing Count': missing_values,
        'Percentage (%)': percentages.round(2)  
    })

    return summary_df

## specifies which columns and row has null values
def specifyWhere(df, columns):
    missing_locations = {}
    for col in columns:
        null_indeces = df[df[col].isnull()].index.tolist()

        if null_indeces:
            missing_locations[col] = null_indeces

    return missing_locations

## if {value} of {column_name} HAS duplicates:
    ## do: 
        ## return the row where the {value} is duplicated
        ## filter all {value} to show all true
        ## show the number of duplicated {value} in a column

def checkSpecificColumnDuplicates(df, column_name):
    duplicate_mask = df.duplicated(subset=[column_name], keep=False) ## checks every value in index to check if {value} has duplicates
    duplicated_rows = df[duplicate_mask] ## passes all duplicates into duplicated_rows
    duplicated_rows = duplicated_rows.dropna(subset=[column_name]) ## drops all values containing {NaN}
    duplicates_clean_view = duplicated_rows[['record_id', column_name]]

    print(f"Total amount of duplicated rows: {len(duplicates_clean_view)}")
    return duplicates_clean_view

def genderRemap(df):
    print("unique:")
    print(f"{df["gender_raw"].unique()}\n")
    clean_source = df["gender_raw"].str.upper().str.strip()

    mapping = {
            r'(?i)^(NON-BINARY|NB)$': 'Non-Binary', 
            r'(?i)^(MALE|M)$': 'Male', 
            r'(?i)^(FEMALE|F)$':'Female', 
            r'(?i)^PREFER NOT TO SAY$': 'Prefer not to say'
            
            }
    
    print(f"dict: {mapping} \n")
    df["gender_raw"] = clean_source.replace(mapping, regex=True)
    
    ## recheck
    print("remapped gender_raw")
    return df["gender_raw"]

def sourceRemap(df):
    print("unique: ")
    print(f"{df["source_system"].unique()}\n")
    clean_source = df["source_system"].str.upper().str.strip()

    mapping = {
        r'(?i)^(STORE|IN-STORE)$': 'In-Store',
        r'(?i)^(MOBILE-APP|MOBILE APP|MOBILE|APP)$': 'Mobile App',
        r'(?i)^(CALL CENTRE|CALL CENTER)$': 'Call Center',
        r'(?i)^(PARTNER-API|PARTNER API)$': 'Partner-API',
        r'(?i)^(MARKETPLACE|MARKET PLACE)$': 'Marketplace',
        r'MARKETPLACE | MARKET PLACE': 'Marketplace',
        r'(?i)^WEB$': 'Web'
    }
    
    print(f"dict: {mapping} \n")
    df["source_system"] = clean_source.replace(mapping, regex=True)

    print("remapped source_system")
    return df["source_system"]


## TODO: 
## INVALID DETECTION (PRIORITY!!!!)
## CITY_RAW
## PRODUCT_CATEGORY_RAW
## CAMPAIGN_SOURCE_RAW
## CUSTOMER_SEGMENT_RAW

def countryRemap(df):
    print("unique: ")
    print(df["country_raw"].unique())
    
    clean_source = df["country_raw"].str.upper().str.strip()
    mapping = {
        r'(?i)^AUS?$|^Australia$': 'Australia',
        r'(?i)^DE$|^Ger(many)?$|^Deutschland$': 'Germany',
        r'(?i)^FR(ance|nce)?$': 'France',
        r'(?i)^SG$|^Singapore$|^S\'pore$': 'Singapore',
        r'(?i)^JP$|^Japan$|^Nippon$': 'Japan',
        r'(?i)^CA(nada|nda)?$': 'Canada',
        r'(?i)^BR(asil|azil)?$': 'Brazil',
        r'(?i)^PH(L|ils\.?)?$|^Philippines$|^Pilipinas$': 'Philippines',
        r'(?i)^US(A|\.S\.A\.)?$|^United States$|^America$': 'United States',
        r'(?i)^CH$|^Swi(tzerland|ss)$|^Suisse$': 'Switzerland'
    }
    
    print(f"dict: {mapping} \n")
    df["country_raw"] = clean_source.replace(mapping, regex=True)

    print("remapped country_raw")
    return df["country_raw"]

def detectInvalidAge(df):
    age_numeric = pd.to_numeric(df["age_raw"], errors='coerce')
    invalid_condition = (age_numeric >= 100) | (age_numeric <= 13) | (age_numeric.isna())
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'age_raw']]

def detectInvalidDiscount(df):
    if df["discount_raw"].dtype == '0':
        mapping = {
        'freebie': '100%',
        'ten percent': '10%'
        }
            
        df["discount_raw"] = df["discount_raw"].replace(mapping, regex=True)
        df["discount_raw"] = df["discount_raw"].str.rstrip('%').astype(float)
        
    discounts = pd.to_numeric(df["discount_raw"], errors='coerce')    
    invalid_condition = (discounts <= 0) | (discounts >= 101) | (discounts.isna())
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'discount_raw']]

def invalidShipOrder(df):
    order_date = pd.to_datetime(df["order_date_raw"], format='mixed', errors='coerce')
    delivery_date = pd.to_datetime(df["delivery_date_raw"], format='mixed', errors='coerce')
    invalid_condition = (order_date > delivery_date)
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'order_date_raw', 'delivery_date_raw']]

def invalidEmail(df): ## confirmed no false positives (look at mateo.nez140@example..com) ((double dots))
    regex = r'^[\w.-]+@([\w-]+\.)+[\w-]{2,4}$'
    is_valid = df['customer_email'].str.contains(regex, regex=True, na=False)
    invalid_email = df[~is_valid]
    return invalid_email[['record_id', 'customer_email']]

def cleanEverything(df):
    genderRemap(df)
    sourceRemap(df)
    countryRemap(df)
## email validation regex: ^[\w-\.]+@([\w-]+\.)+[\w-]{2,4}$     

In [45]:
invalidEmail(df)

C:\Users\tomas\AppData\Local\Temp\ipykernel_14208\2630652328.py:153: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_valid = df['customer_email'].str.contains(regex, regex=True, na=False)


,record_id,customer_email
3,GM-00347,mateo.nez140@example..com
4,GM-01214,maria.davis@companycom
5,GM-00794,chlo.kim@companycom
9,GM-01241,siti.dubois675@example..com
13,GM-01224,zo.garca at hotmail.com
...,...,...
1964,GM-01200,c.brown874@example..com
1984,GM-01781,NaN
1988,GM-00302,jay.gonzales946@example..com
1993,GM-00838,jay.nez920@example..com
